# Paso 6: Revisar balance del dataset

Este notebook cuenta cuántas subcarpetas (clases) tiene tu dataset y cuántas imágenes hay en cada una para detectar desbalance.

In [1]:
from pathlib import Path

DATASET_DIR = Path(r"C:\Users\jaevi\Downloads\Datasets\dataset curado\dataset_consolidado\version1")

# Solo estas carpetas (en este orden)
TARGET_CLASSES = [
    "early_blight",
    "pest",
    "mosaic_virus",
    "leafroll_virus",
    "healthy",
    "late_blight",
    "nematode",
]

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

print(f"Ruta seleccionada: {DATASET_DIR}")
print(f"Existe: {DATASET_DIR.exists()}")

Ruta seleccionada: C:\Users\jaevi\Downloads\Datasets\dataset curado\dataset_consolidado\version1
Existe: True


In [2]:
import pandas as pd

if not DATASET_DIR.exists() or not DATASET_DIR.is_dir():
    raise FileNotFoundError("La ruta no existe o no es una carpeta.")

# Solo carpetas objetivo
class_dirs = [DATASET_DIR / cls for cls in TARGET_CLASSES if (DATASET_DIR / cls).is_dir()]
missing = [cls for cls in TARGET_CLASSES if not (DATASET_DIR / cls).is_dir()]

if missing:
    print("Aviso: estas carpetas no existen y se omiten:", missing)

if not class_dirs:
    raise ValueError("No se encontraron carpetas objetivo dentro del dataset.")

rows = []
for class_dir in class_dirs:
    count = sum(
        1
        for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    rows.append({"clase": class_dir.name, "imagenes": count})

# Resumen por clase
summary_df = pd.DataFrame(rows).sort_values("imagenes", ascending=False).reset_index(drop=True)
total_images = int(summary_df["imagenes"].sum())
num_classes = int(len(summary_df))
summary_df["porcentaje"] = (summary_df["imagenes"] / total_images * 100).round(2)

max_count = int(summary_df["imagenes"].max())
min_count = int(summary_df["imagenes"].min())
imbalance_ratio = (max_count / min_count) if min_count > 0 else float("inf")

print("=" * 60)
print("RESUMEN DEL DATASET")
print("=" * 60)
print(f"Carpeta analizada: {DATASET_DIR}")
print(f"Numero de clases (subcarpetas): {num_classes}")
print(f"Total de imagenes: {total_images}")
print(f"Mayor clase: {max_count} imagenes")
print(f"Menor clase: {min_count} imagenes")
print(f"Ratio max/min: {imbalance_ratio:.2f}")
print("=" * 60)
print("\nImagenes por clase:")
display(summary_df)

# Diagnostico simple
if min_count == 0:
    print("\nDiagnostico: dataset fuertemente desbalanceado (hay clases con 0 imagenes).")
elif imbalance_ratio <= 1.5:
    print("\nDiagnostico: dataset balanceado o casi balanceado.")
elif imbalance_ratio <= 3:
    print("\nDiagnostico: desbalance moderado.")
else:
    print("\nDiagnostico: desbalance alto.")

RESUMEN DEL DATASET
Carpeta analizada: C:\Users\jaevi\Downloads\Datasets\dataset curado\dataset_consolidado\version1
Numero de clases (subcarpetas): 7
Total de imagenes: 9665
Mayor clase: 2981 imagenes
Menor clase: 617 imagenes
Ratio max/min: 4.83

Imagenes por clase:


,clase,imagenes,porcentaje
0,early_blight,2981,30.84
1,late_blight,2525,26.13
2,healthy,1218,12.60
3,pest,857,8.87
4,nematode,805,8.33
5,mosaic_virus,662,6.85
6,leafroll_virus,617,6.38



Diagnostico: desbalance alto.
